In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/LeBlanc2022_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["JK124", "JK125", "JK126", "JK134", "JK136", "JK142", "JK152", "JK153", "JK156", "JK163"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 34000 × 19849
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [8]:
#adata = adata.raw.to_adata()

In [9]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 4000/4000 [00:06<00:00, 643.82it/s]


In [10]:
adata.X = X_counts_recovered

In [11]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [12]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [13]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [14]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [15]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF470-DT,False,1067,3.138235,True,0.014693,0.008692,0.632066
AC092667.2,False,256,0.752941,True,0.003242,0.001756,0.555654
ZNF367,False,2127,6.255882,True,0.025442,0.012582,0.472065
SULT1B1,False,249,0.732353,True,0.002998,0.001545,0.549887
TRIM63,False,45,0.132353,True,0.000627,0.000389,0.601799
...,...,...,...,...,...,...,...
LINC02318,False,20,0.058824,True,0.000259,0.000151,0.594276
LINC01807,False,32,0.094118,True,0.000378,0.000191,0.498490
LINC02338,False,20,0.058824,True,0.000251,0.000129,0.528389


In [16]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [17]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [18]:
adata.var = df_tmp

In [19]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [20]:
adata

View of AnnData object with n_obs × n_vars = 34000 × 16557
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [21]:
adata.obs['donor_id'] = df_obs['donor_id']

In [22]:
metadata_data = {
    'Author': ['LeBlanc2022'] * 10,
    'donor_id': ["JK124", "JK125", "JK126", "JK134", "JK136", "JK142", "JK152", "JK153", "JK156", "JK163"],
    'stage': ['Primary'] * 10,
    'assay': ['10x 3\' v3'] * 10,
    'tissue': [
        'right temporal lobe', 'right frontal lobe', 'left frontal lobe', 'right frontal lobe', 'left temporal lobe',
        'right temporal lobe', 'right parietal lobe', 'left parietaloccipital', 'left temporal lobe', 'right frontal lobe'
    ],
    'Cells': ['Total'] * 10,
    'Method': ['cell'] * 10
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

        Author donor_id    stage      assay                  tissue  Cells  \
0  LeBlanc2022    JK124  Primary  10x 3' v3     right temporal lobe  Total   
1  LeBlanc2022    JK125  Primary  10x 3' v3      right frontal lobe  Total   
2  LeBlanc2022    JK126  Primary  10x 3' v3       left frontal lobe  Total   
3  LeBlanc2022    JK134  Primary  10x 3' v3      right frontal lobe  Total   
4  LeBlanc2022    JK136  Primary  10x 3' v3      left temporal lobe  Total   
5  LeBlanc2022    JK142  Primary  10x 3' v3     right temporal lobe  Total   
6  LeBlanc2022    JK152  Primary  10x 3' v3     right parietal lobe  Total   
7  LeBlanc2022    JK153  Primary  10x 3' v3  left parietaloccipital  Total   
8  LeBlanc2022    JK156  Primary  10x 3' v3      left temporal lobe  Total   
9  LeBlanc2022    JK163  Primary  10x 3' v3      right frontal lobe  Total   

  Method  
0   cell  
1   cell  
2   cell  
3   cell  
4   cell  
5   cell  
6   cell  
7   cell  
8   cell  
9   cell  


In [23]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id       Author    stage      assay               tissue  Cells  \
0        JK124  LeBlanc2022  Primary  10x 3' v3  right temporal lobe  Total   
1        JK124  LeBlanc2022  Primary  10x 3' v3  right temporal lobe  Total   
2        JK124  LeBlanc2022  Primary  10x 3' v3  right temporal lobe  Total   
3        JK124  LeBlanc2022  Primary  10x 3' v3  right temporal lobe  Total   
4        JK124  LeBlanc2022  Primary  10x 3' v3  right temporal lobe  Total   
...        ...          ...      ...        ...                  ...    ...   
33995    JK163  LeBlanc2022  Primary  10x 3' v3   right frontal lobe  Total   
33996    JK163  LeBlanc2022  Primary  10x 3' v3   right frontal lobe  Total   
33997    JK163  LeBlanc2022  Primary  10x 3' v3   right frontal lobe  Total   
33998    JK163  LeBlanc2022  Primary  10x 3' v3   right frontal lobe  Total   
33999    JK163  LeBlanc2022  Primary  10x 3' v3   right frontal lobe  Total   

      Method  
0       cell  
1       cell  
2     

In [24]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [25]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
AAACCTGCACAACTGT-1-0-1,JK124,1785,1774.673828,Neoplastic,Differentiated-like,MES-like,Astrocyte,Fibroblasts,malignant cell
AAACCTGCACCAGCAC-1-0-1,JK124,1870,1692.728516,Neoplastic,Differentiated-like,MES-like,Astrocyte,Fibroblasts,malignant cell
AAACCTGGTCAGGACA-1-0-1,JK124,1438,1688.208618,Neoplastic,Differentiated-like,MES-like,Astrocyte,Fibroblasts,malignant cell
AAACGGGAGCTGCCCA-1-0-1,JK124,4442,2041.248535,Neoplastic,Stem-like,OPC-like,Neuron,Pluripotent Stem Cells,malignant cell
AAACGGGTCTGTGCAA-1-0-1,JK124,2632,1884.636230,Neoplastic,Differentiated-like,AC-like,Microglial cell,Radial Glia Cells,malignant cell
...,...,...,...,...,...,...,...,...,...
TTGTAGGGTAAATGTG-96-0-1,JK163,3284,2160.783447,Neoplastic,Stem-like,OPC-like,Oligodendrocyte,Neurons,malignant cell
TTTACTGAGTGTTTGC-96-0-1,JK163,3285,2094.651855,Neoplastic,Differentiated-like,AC-like,Astrocyte,Fibroblasts,malignant cell
TTTATGCAGCACAGGT-96-0-1,JK163,1413,1493.527954,Neoplastic,Differentiated-like,AC-like,Astrocyte,Fibroblasts,malignant cell
TTTATGCCAAAGCAAT-96-0-1,JK163,2818,1912.695068,Neoplastic,Stem-like,OPC-like,Neuron,Pluripotent Stem Cells,malignant cell


In [26]:
merged_obs_df.index= df_obs.index

In [27]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [28]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [29]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [30]:
del merged_obs_df['donor_id_y']

In [31]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [32]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [33]:
adata.obs = merged_obs_df

In [34]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    643 total control genes are used. (0:00:01)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    510 total control genes are used. (0:00:01)
-->     'phase', cell cycle phase (adata.obs)


In [35]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/LeBlanc2022_Part3.h5ad")